# 🔬 Multi-Agent Research System

A portfolio-ready Generative AI project that coordinates multiple specialized agents to research a question, collect evidence, fact-check findings, and synthesize a structured final report.

### Agent Architecture
**User Query → Planner → Specialist Researchers → Evidence Aggregator → Fact Checker → Synthesizer → Research Report**

This notebook uses an OpenRouter-compatible OpenAI client, while also including a local demo corpus so the workflow can be explored without an API key.


In [ ]:
%pip install -q openai python-dotenv pydantic


In [ ]:
import os
import json
from typing import List, Dict
from pydantic import BaseModel, Field

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

print("API configured:", bool(OPENROUTER_API_KEY))
print("Model:", OPENROUTER_MODEL)


In [ ]:
class ResearchTask(BaseModel):
    task_id: str
    role: str
    question: str
    focus: str

class Evidence(BaseModel):
    source: str
    claim: str
    evidence: str
    confidence: float = Field(ge=0, le=1)

class AgentResult(BaseModel):
    agent: str
    summary: str
    evidence: List[Evidence]
    limitations: List[str] = []

def make_task(task_id, role, question, focus):
    return ResearchTask(
        task_id=task_id,
        role=role,
        question=question,
        focus=focus
    )

print("Research data models ready.")


## 🧠 1. Planner Agent

The planner decomposes a broad research question into focused tasks. In a production system, this step can use an LLM and structured output validation.


In [ ]:
def planner_agent(question: str) -> List[ResearchTask]:
    return [
        make_task("T1", "Technical Researcher", question,
                  "technology, architecture, capabilities, implementation"),
        make_task("T2", "Evidence Researcher", question,
                  "evidence, measurable facts, examples, limitations"),
        make_task("T3", "Industry Researcher", question,
                  "real-world adoption, use cases, risks, business impact"),
    ]

question = "How can multi-agent AI systems improve research workflows?"
tasks = planner_agent(question)

for task in tasks:
    print(f"{task.task_id} | {task.role} | {task.focus}")


## 📚 2. Demo Research Corpus

This small corpus makes the notebook runnable without live web-search credentials. Replace it with web search, company documents, academic papers, or a RAG vector database in production.


In [ ]:
DEMO_CORPUS = [
    {
        "source": "Architecture Notes",
        "text": "Multi-agent systems divide complex work among specialized agents. A planner can decompose a request while specialist agents focus on separate research dimensions."
    },
    {
        "source": "Evaluation Notes",
        "text": "Research agents should be evaluated for factual accuracy, source quality, citation correctness, task completion, latency, and cost."
    },
    {
        "source": "Reliability Notes",
        "text": "Independent evidence gathering and a dedicated verification stage can reduce unsupported claims, but multi-agent systems still require source validation and human review for high-impact research."
    },
    {
        "source": "Production Notes",
        "text": "Graph-based orchestration can coordinate planning, parallel research, verification, retries, and final synthesis while maintaining state between steps."
    },
]

def search_corpus(query: str, top_k: int = 3):
    terms = {word.lower().strip(".,?!") for word in query.split() if len(word) > 3}
    scored = []
    for item in DEMO_CORPUS:
        text = item["text"].lower()
        score = sum(term in text for term in terms)
        scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [item for score, item in scored[:top_k] if score > 0]

print("Corpus documents:", len(DEMO_CORPUS))


## 🔎 3. Specialist Research Agents

Each specialist receives one focused task and produces evidence. The example below uses the local corpus; an LLM-powered implementation can turn retrieved passages into structured evidence.


In [ ]:
def specialist_researcher(task: ResearchTask) -> AgentResult:
    docs = search_corpus(task.focus + " " + task.question)

    evidence = []
    for doc in docs:
        evidence.append(
            Evidence(
                source=doc["source"],
                claim=f"Relevant finding for {task.role}",
                evidence=doc["text"],
                confidence=0.85
            )
        )

    summary = (
        f"{task.role} investigated the question from the perspective of "
        f"{task.focus}. Retrieved {len(evidence)} relevant evidence items."
    )

    return AgentResult(
        agent=task.role,
        summary=summary,
        evidence=evidence,
        limitations=["Demo corpus is intentionally small and is not live web research."]
    )

results = [specialist_researcher(task) for task in tasks]

for result in results:
    print("\n", result.agent)
    print(result.summary)


## 🧩 4. Evidence Aggregator

The aggregator combines outputs from independent researchers into a common evidence pool.


In [ ]:
def aggregate_evidence(results: List[AgentResult]) -> List[Evidence]:
    evidence = []
    for result in results:
        evidence.extend(result.evidence)

    unique = {}
    for item in evidence:
        key = (item.source, item.evidence)
        unique[key] = item

    return list(unique.values())

evidence_pool = aggregate_evidence(results)

print("Unique evidence items:", len(evidence_pool))
for item in evidence_pool:
    print(f"- [{item.source}] confidence={item.confidence:.2f}: {item.evidence}")


## ✅ 5. Fact-Checker Agent

The fact-checker identifies unsupported or weakly supported claims before synthesis. In production, this stage can compare claims against retrieved source passages and ask an LLM to return structured verification results.


In [ ]:
def fact_checker(evidence: List[Evidence]) -> Dict:
    verified = [item for item in evidence if item.confidence >= 0.80]
    weak = [item for item in evidence if item.confidence < 0.80]

    return {
        "verified_count": len(verified),
        "weak_count": len(weak),
        "verified_sources": [item.source for item in verified],
        "warnings": [
            "Demo evidence is synthetic and should not be treated as authoritative."
        ] if evidence else ["No evidence was retrieved."]
    }

fact_check = fact_checker(evidence_pool)
print(json.dumps(fact_check, indent=2))


## ✍️ 6. Synthesizer Agent

The synthesizer turns verified evidence into a concise research report with explicit source attribution and limitations.


In [ ]:
def synthesizer(question: str, results: List[AgentResult], evidence: List[Evidence], check: Dict) -> str:
    lines = [
        "# Multi-Agent Research Report",
        "",
        f"## Research Question\n{question}",
        "",
        "## Key Findings",
    ]

    for idx, item in enumerate(evidence, 1):
        lines.append(
            f"{idx}. **{item.claim}** — {item.evidence} "
            f"_(Source: {item.source}; confidence: {item.confidence:.2f})_"
        )

    lines.extend([
        "",
        "## Agent Perspectives",
    ])

    for result in results:
        lines.append(f"- **{result.agent}:** {result.summary}")

    lines.extend([
        "",
        "## Reliability Notes",
        f"- Verified evidence items: {check['verified_count']}",
        f"- Weak evidence items: {check['weak_count']}",
        f"- Warning: {check['warnings'][0]}",
    ])

    return "\n".join(lines)

report = synthesizer(question, results, evidence_pool, fact_check)
print(report)


## 🤖 7. Optional OpenRouter LLM Layer

Use this helper when an `OPENROUTER_API_KEY` is available. The same agent architecture can then use an LLM for planning, research interpretation, verification, and synthesis.

Keep API keys in environment variables and never commit `.env` files to GitHub.


In [ ]:
def llm_chat(system_prompt: str, user_prompt: str) -> str:
    if not OPENROUTER_API_KEY:
        return "No API key configured. Use the local demo workflow or configure OPENROUTER_API_KEY."

    from openai import OpenAI

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.choices[0].message.content

if OPENROUTER_API_KEY:
    print(llm_chat(
        "You are a concise research assistant.",
        "Explain in two sentences why evidence verification matters in AI research."
    ))
else:
    print("Configure OPENROUTER_API_KEY to enable live LLM calls.")


## 🚀 8. End-to-End Multi-Agent Pipeline

This function demonstrates the complete orchestration pattern:

1. Planner creates specialist tasks.
2. Researchers investigate independently.
3. Evidence is aggregated.
4. Fact-checker validates evidence quality.
5. Synthesizer creates the final report.


In [ ]:
def multi_agent_research(question: str):
    tasks = planner_agent(question)
    results = [specialist_researcher(task) for task in tasks]
    evidence = aggregate_evidence(results)
    check = fact_checker(evidence)
    report = synthesizer(question, results, evidence, check)

    return {
        "tasks": tasks,
        "agent_results": results,
        "evidence": evidence,
        "fact_check": check,
        "report": report,
    }

output = multi_agent_research("What are the benefits and risks of multi-agent AI research systems?")
print(output["report"])


## 🏗️ 9. Production Architecture

A production version can evolve into:

- **LangGraph** — stateful graph orchestration, branching, retries, checkpoints
- **LangChain** — model/tool integrations and structured outputs
- **MCP** — standardized access to search, files, databases, and external tools
- **RAG** — vector database for internal documents and trusted sources
- **Web Search** — live research with source URLs and publication dates
- **Redis** — caching and short-term agent state
- **PostgreSQL / MongoDB** — research jobs, reports, metadata, and audit history
- **Observability** — LangSmith, OpenTelemetry, Prometheus, or Grafana
- **Human Review** — approval gate for high-impact research

### Suggested production graph

`START → Planner → Parallel Researchers → Evidence Merge → Fact Checker → Citation Validator → Synthesizer → Quality Gate → END`

The parallel researcher branches are especially useful because technical, market, academic, and risk research can run independently before synthesis.


## 📊 10. Evaluation Metrics

Track these metrics when turning the notebook into a serious AI engineering project:

| Metric | What it measures |
|---|---|
| Citation Accuracy | Whether citations actually support claims |
| Factuality | Percentage of claims supported by evidence |
| Research Coverage | Important subtopics successfully investigated |
| Source Quality | Reliability and authority of sources |
| Task Completion | Whether the requested research was completed |
| Latency | Total and per-agent execution time |
| Cost | Token/API cost per research job |
| Consistency | Stability across repeated runs |
| Safety | Resistance to prompt injection and unsafe tool use |



## 🛡️ 11. Guardrails and Reliability

For a production system, add:

- Input validation and prompt-injection detection
- Tool permission boundaries
- Domain allowlists for sensitive research
- Source freshness checks
- Citation verification
- Structured JSON outputs with Pydantic
- Retry limits and timeouts
- Human approval for sensitive decisions
- Complete agent traces and audit logs
- Secrets stored outside source control

### 🔥 Portfolio Upgrade Ideas

1. Add real web search.
2. Add a RAG knowledge base.
3. Run specialist agents in parallel with LangGraph.
4. Add MCP tools for search and document access.
5. Add a citation validator.
6. Add a Streamlit or React dashboard.
7. Store research history in MongoDB/PostgreSQL.
8. Add evaluation datasets and automated scoring.
9. Deploy the API and UI to a cloud platform.
10. Add human-in-the-loop review.


## 🎯 Learning Outcomes

By completing this project, you practice:

- Multi-agent architecture
- Agent specialization and task decomposition
- Evidence-based generation
- Structured LLM outputs
- Fact checking and citation-aware synthesis
- Agent orchestration
- RAG and tool integration
- MCP-based tool access
- Evaluation and observability
- Production GenAI system design

**Project goal:** move from a single LLM chatbot to a coordinated, reliable AI research team.
